# RSW / 01 — Regime-switching AR(1) (discretized regime-switching OU) on MNQ 5-minute returns: discovery 2020-01 – 2023-06

Protocol: [`RSW 001`](../../discussions/MNQ_Regime_Switching/001_claude_rsw01_preregistration.md). Results in ticks per contract (1 tick = 0.25 points = $0.50).

z_t = c_s + φ_s z_{t−1} + σ_s ε_t with a hidden Markov state s. φ_s < 0 means reversal and φ_s > 0 continuation. M1/M2 have 2/3 states with constant transition probabilities. M3/M4 let the transition probabilities depend on the previous bar's order flow (volume surprise, |BVC imbalance|, imbalance × sign z, from the 1-minute bars). M0 is the one-state AR(1) benchmark.

- **R1**: out-of-sample predictive log score beats M0 (Holm ≤ 0.05).
- **R2**: φ_max − φ_min ≥ 0.05 with t ≥ 3 in every fit.
- **R3**: extreme states have p_stay ≥ 2/3 in every fit.
- **R4**: R²_OOS > 0 against zero and against M0, Clark–West against M0 (Holm ≤ 0.05).
- **E**: one-bar trade when the forecast move is ≥ 3 ticks; net > 0 (Holm ≤ 0.05), ≥ 150 trades, positive in ≥ 2 of 3 periods.

In [ ]:
import json, os, sys, tempfile, pickle
import datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SYNTHETIC = os.environ.get("RSW_SYNTHETIC") == "1"
if SYNTHETIC:   # own folder: planted 2-state MS-AR(1), phi = -0.3 / +0.3, switching driven by volume surprise
    os.environ["EQUITY_TRADING_DIR"] = os.environ.get("RSW_SYNTH_DIR") or tempfile.mkdtemp(prefix="rsw_")
    os.environ["RSW_SYNTH_DIR"] = os.environ["EQUITY_TRADING_DIR"]

REPO = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(REPO / "src"))
from equity_trading import mnq_predict as P
from equity_trading import reversal as V
from equity_trading import regime_ar as R
from equity_trading import micro_bars as MB

CFG = P.SYNTH_CFG if SYNTHETIC else P.MRPConfig()
RC = R.RSWConfig()
D = dt.date
OUT = Path("01"); OUT.mkdir(exist_ok=True)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 80)
print("engines", R.RSW_VERSION, P.MRP_VERSION, V.REV_VERSION, "| synthetic" if SYNTHETIC else "| real data")
if SYNTHETIC and not (Path(os.environ["EQUITY_TRADING_DIR"]) / "Databento").exists():
    NULL = os.environ.get("RSW_SYNTH_NULL") == "1"     # null smoke: no regimes, no flow effect (false-positive check)
    R.write_synthetic_regimes(os.environ["EQUITY_TRADING_DIR"], CFG.warmup_start, CFG.train[1], CFG,
                              **({"phis": (0.0, 0.0), "flow_effect": 0.0} if NULL else {}))

In [ ]:
if SYNTHETIC:
    FIRST_FIT, REFITS, END = (CFG.train[0], D(2020, 7, 1)), CFG.refits_train, CFG.train[1]
    PERIODS = {"2020Q3": (D(2020, 7, 1), D(2020, 9, 30)), "2020Q4": (D(2020, 10, 1), D(2020, 12, 31))}
else:
    FIRST_FIT, REFITS, END = (CFG.train[0], D(2021, 1, 1)), CFG.refits_train, CFG.train[1]
    PERIODS = {"2021": (D(2021, 1, 1), D(2021, 12, 31)), "2022": (D(2022, 1, 1), D(2022, 12, 31)),
               "2023H1": (D(2023, 1, 1), D(2023, 6, 30))}
NEED_PERIODS = int(np.ceil(2 / 3 * len(PERIODS)))
grid = P.load_grid(CFG.warmup_start, END, CFG)
sig2 = V.event_sigma(grid, FIRST_FIT, REFITS, END, CFG)
m1 = MB.load_1m(CFG.root, CFG.warmup_start - dt.timedelta(days=3), END + dt.timedelta(days=1))
S = R.build_series(grid, sig2, m1, CFG, RC)
S = S[(S["trade_date"] >= CFG.train[0]) & (S["trade_date"] <= END)]
del m1
td = S["trade_date"].to_numpy()
OOS = td >= REFITS[0]
o = S["obs"].to_numpy()
print(f"{len(S):,} bars | observed {o.sum():,} | out-of-sample observed {(o & OOS).sum():,} | CME days {S['trade_date'].nunique()}")
cov = pd.DataFrame({
    "eligible": S["eligible"].mean(), "vsurp defined": np.isfinite(S["vsurp"]).mean(),
    "imb defined": np.isfinite(S["imb"]).mean(), "obs": o.mean(),
    "corr(imb, z)": np.corrcoef(S.loc[o, "imb"], S.loc[o, "z"])[0, 1],
    "corr(vsurp, |z|)": np.corrcoef(S.loc[o, "vsurp"], S.loc[o, "z"].abs())[0, 1],
    "lag-1 autocorr of z (obs, same day)": np.corrcoef(S.loc[o & S["has_lag"], "z"], S.loc[o & S["has_lag"], "zl"])[0, 1]},
    index=["value"]).T
display(cov.round(4))
zv = S[o].groupby(pd.to_datetime(S.loc[o, "trade_date"]).dt.to_period("Q"))["z"].var()
print("var(z) by quarter (1 = volatility model level right):"); display(zv.round(3).to_frame().T)

## 1. Walk-forward fits (expanding, frozen per segment, causal filter)

In [ ]:
MODELS = [("M0", 1, False)] + list(RC.models)
cache = OUT / f"fits_{R.RSW_VERSION}{'_synth' if SYNTHETIC else ''}.pkl"
if cache.exists():
    FC, FITS = pickle.load(open(cache, "rb"))
    print("loaded", cache)
else:
    FC, FITS = {}, {}
    for name, K, ex in MODELS:
        t0 = dt.datetime.now()
        FC[name], FITS[name] = R.walk_forward(S, CFG.train[0], REFITS, END, K, ex, RC)
        print(name, "K", K, "flow TVTP", ex, "| iterations", [f.n_iter for f in FITS[name]],
              "converged", [f.converged for f in FITS[name]], "|", dt.datetime.now() - t0)
    pickle.dump((FC, FITS), open(cache, "wb"))
fit_rows = []
for name, _, _ in MODELS:
    for a, f in zip(REFITS, FITS[name]):
        fit_rows.append({"model": name, "fit_before": a, "K": f.K, "loglik": f.loglik, "bic": f.bic,
                         "n_obs": f.n_obs, "iter": f.n_iter, "converged": f.converged})
fits_df = pd.DataFrame(fit_rows)
display(fits_df.round(1))

In [ ]:
for name, _, _ in MODELS:
    for a, f in zip(REFITS, FITS[name]):
        print(f"--- {name}, fit on data before {a}")
        display(f.table().round(4))

## 2. Gates R1–R4

In [ ]:
Sx = S[OOS]
day = Sx["trade_date"].to_numpy()
z_oos = np.where(Sx["obs"], Sx["z"], np.nan)
lp0 = FC["M0"]["lpz"].reindex(Sx.index).to_numpy()
fc0 = FC["M0"]["fc"].reindex(Sx.index).to_numpy()
rows = []
for name, K, ex in RC.models:
    f = FC[name].reindex(Sx.index)
    dm = R.predictive_dm(f["lpz"].to_numpy(), lp0, day, RC.nw_lags)
    gaps = [R.phi_gap(m) for m in FITS[name]]
    sc = R.forecast_scores(z_oos, f["fc"].to_numpy(), fc0, day, RC.nw_lags)
    rows.append({"model": name, "K": K, "flow_tvtp": ex,
                 "logscore_gain_per_bar": dm["per_bar_diff"], "R1_t": dm["t"], "R1_p": dm["p"],
                 "phi_min": [round(g["phi_min"], 3) for g in gaps], "phi_max": [round(g["phi_max"], 3) for g in gaps],
                 "t_gap_min": min(g["t_gap"] for g in gaps),
                 "R2": all(g["gap"] >= RC.phi_gap and g["t_gap"] >= RC.t_gap for g in gaps),
                 "stay_min": min(min(g["stay_min_state"], g["stay_max_state"]) for g in gaps),
                 "R3": all(min(g["stay_min_state"], g["stay_max_state"]) >= RC.min_stay for g in gaps),
                 "r2_vs_zero": sc["r2_oos_vs_zero"], "r2_vs_M0": sc["r2_oos_vs_M0"], "CW_t": sc["cw_t"],
                 "CW_p": sc["cw_p"], "hit_rate": sc["hit_rate"]})
G = pd.DataFrame(rows).set_index("model")
G["R1_p_holm"] = pd.Series(R.holm(G["R1_p"].to_dict()))
G["R1"] = (G["logscore_gain_per_bar"] > 0) & (G["R1_p_holm"] <= RC.alpha)
G["CW_p_holm"] = pd.Series(R.holm(G["CW_p"].to_dict()))
G["R4"] = (G["r2_vs_zero"] > 0) & (G["r2_vs_M0"] > 0) & (G["CW_p_holm"] <= RC.alpha)
m0s = R.forecast_scores(z_oos, fc0, np.zeros_like(fc0), day)
print(f"M0 (plain AR(1)): R2_OOS vs zero {m0s['r2_oos_vs_zero']:.5f}, phi per fit",
      [round(float(m.phi[0]), 4) for m in FITS["M0"]])
display(G.T)

## 3. Gate E: one-bar trade in the sign of the forecast (ticks per contract)

In [ ]:
erows, TR = [], {}
for name, _, _ in RC.models:
    fc = FC[name]["fc"].reindex(Sx.index).to_numpy()
    for thr, cost, lab in ((RC.cost_ticks, RC.cost_ticks, "gate: thr 3, cost 3"),
                           (RC.realized_cost_ticks, RC.realized_cost_ticks, "thr 1.6, cost 1.6"),
                           (0.0, 0.0, "every bar, gross")):
        tr = R.one_bar_trades(Sx, fc, thr)
        TR[(name, lab)] = tr
        erows.append({"model": name, "rule": lab, **R.trade_stats(tr, cost, PERIODS)})
E = pd.DataFrame(erows).set_index(["model", "rule"])
gate = E.xs("gate: thr 3, cost 3", level="rule").copy()
gate["p_holm"] = pd.Series(R.holm(gate["p"].fillna(1.0).to_dict()))
gate["periods_pos"] = sum((gate[f"net_{p}"] > 0).astype(int) for p in PERIODS)
gate["E"] = (gate["trades"] >= RC.min_trades) & (gate["net"] > 0) & (gate["p_holm"] <= RC.alpha) & \
    (gate["periods_pos"] >= NEED_PERIODS)
display(E.round(3))
display(gate[["trades", "gross", "net", "t", "p_holm", "periods_pos", "E"]].round(3))

## 4. Descriptive: what the regimes are (filtered argmax state, out of sample)

In [ ]:
for name, K, ex in RC.models:
    f = FC[name].reindex(Sx.index)
    P_ = f[[f"p{k}" for k in range(K)]].to_numpy()
    ok = Sx["obs"].to_numpy() & np.isfinite(P_).all(1)
    st = P_[ok].argmax(1)
    sx = Sx[ok]
    q = pd.qcut(np.sqrt(sx["sig2"]), 5, labels=[f"Q{i}" for i in range(1, 6)])
    print(f"=== {name}: share of bars in each state (states ordered by phi: 0 = most reverting)")
    tab = pd.concat({"all": pd.Series(st).value_counts(normalize=True).sort_index(),
                     **{s_: pd.Series(st[sx["session"].to_numpy() == s_]).value_counts(normalize=True).sort_index()
                        for s_ in P.SESSIONS},
                     **{str(y): pd.Series(st[pd.to_datetime(sx["trade_date"]).dt.year.to_numpy() == y]).value_counts(normalize=True).sort_index()
                        for y in sorted(pd.to_datetime(sx["trade_date"]).dt.year.unique())},
                     **{f"sigma {c}": pd.Series(st[(q == c).to_numpy()]).value_counts(normalize=True).sort_index()
                        for c in q.cat.categories}}, axis=1).fillna(0)
    display(tab.round(3))
    # realized lag-1 autocorrelation of z inside each filtered state: does the state label mean what phi says?
    hl = sx["has_lag"].to_numpy()
    ac = {k: np.corrcoef(sx["z"].to_numpy()[hl & (st == k)], sx["zl"].to_numpy()[hl & (st == k)])[0, 1]
          for k in range(K) if (hl & (st == k)).sum() > 100}
    print("out-of-sample lag-1 autocorr of z by filtered state:", {k: round(v, 4) for k, v in ac.items()})
    sm = R.smooth(S, FITS[name][-1])[OOS][ok].argmax(1)
    print(f"filtered vs smoothed state agreement (last fit): {np.mean(sm == st):.3f}")

In [ ]:
best = G["logscore_gain_per_bar"].idxmax()
K = dict((n, k) for n, k, _ in RC.models)[best]
f = FC[best].reindex(Sx.index)
days = sorted(Sx["trade_date"].unique())
week = [d for d in days if days[len(days) // 2] <= d < days[len(days) // 2] + dt.timedelta(days=7)]
sel = np.isin(Sx["trade_date"].to_numpy(), week)
fig, ax = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
ax[0].plot(Sx.index[sel], Sx["close"].to_numpy()[sel], lw=0.8, color="k"); ax[0].set_ylabel("MNQ close")
for k in range(K):
    ax[1].plot(Sx.index[sel], f[f"p{k}"].to_numpy()[sel], lw=0.8,
               label=f"state {k} (phi {FITS[best][-1].phi[k]:+.3f})")
ax[1].set_ylabel("P(state | F_t)"); ax[1].legend(loc="upper right", fontsize=8)
ax[0].set_title(f"{best}: filtered regime probabilities, sample week")
plt.tight_layout(); plt.savefig(OUT / "sample_week.png", dpi=110); plt.show()

fig, ax = plt.subplots(1, len(RC.models), figsize=(16, 3.5), sharey=True)
for i, (name, K, _) in enumerate(RC.models):
    for k in range(K):
        ax[i].plot([str(a) for a in REFITS], [m.phi[k] for m in FITS[name]], marker="o", label=f"state {k}")
    ax[i].axhline(0, color="grey", lw=0.6); ax[i].set_title(name); ax[i].legend(fontsize=7)
ax[0].set_ylabel("phi by fit")
plt.tight_layout(); plt.savefig(OUT / "phi_by_fit.png", dpi=110); plt.show()

## 5. Verdict

In [ ]:
V_ = G[["R1", "R2", "R3", "R4"]].join(gate[["E"]])
V_["verdict"] = np.where(V_.all(axis=1), "CANDIDATE (2026 pre-registration)",
                np.where(V_[["R1", "R2", "R3"]].all(axis=1), "REGIMES_NO_TRADABLE_DIRECTION",
                np.where(V_["R1"], "R1_ONLY (volatility residue unless R2)", "NO_REGIME_STRUCTURE")))
display(V_)
summary = {"study": "RSW 01", "engine": R.RSW_VERSION, "synthetic": SYNTHETIC,
           "oos": [str(REFITS[0]), str(END)], "periods": {k: [str(a), str(b)] for k, (a, b) in PERIODS.items()},
           "coverage": {k: float(v) for k, v in cov["value"].items()},
           "fits": json.loads(fits_df.to_json(orient="records", date_format="iso")),
           "params": {n: [FITS[n][i].table().reset_index().to_dict(orient="records") for i in range(len(REFITS))]
                      for n, _, _ in MODELS},
           "gates_R": json.loads(G.drop(columns=["phi_min", "phi_max"]).to_json(orient="index")),
           "phi": {n: {"min": G.loc[n, "phi_min"], "max": G.loc[n, "phi_max"]} for n in G.index},
           "trades": json.loads(E.reset_index().to_json(orient="records")),
           "gate_E": json.loads(gate.to_json(orient="index")),
           "verdict": V_["verdict"].to_dict()}
json.dump(summary, open(OUT / "summary.json", "w"), indent=1, default=str)
print("written", OUT / "summary.json")